# NeuroGolf Team Validator — shared notebook
Use this notebook in every teammate/GPT session. Update the configuration cell, then run only the sections you need.

In [ ]:
# Run once per environment
%pip install -q numpy onnx onnxruntime


In [ ]:
from pathlib import Path
import sys

TOOL_DIR = Path('.')  # folder containing ngolf_validator.py
sys.path.insert(0, str(TOOL_DIR.resolve()))
import ngolf_validator as ng

BASELINE_ZIP = Path('submission_best.zip')
CANDIDATE_ZIP = Path('submission_candidate.zip')
DATA_DIR = Path('neurogolf_2026_data')
DATA_ZIP = Path('neurogolf-2026.zip')
BASELINE_SCORE = 7982.53  # update every session
TASK = 101
RANDOM_CASES = 1000

ng.ensure_data_dir(DATA_DIR, DATA_ZIP)
print('Ready. Baseline SHA:', ng.sha256_file(BASELINE_ZIP))


## Audit the baseline archive

In [ ]:
audit = ng.audit_submission_zip(BASELINE_ZIP)
audit


## Validate one candidate model against the baseline

In [ ]:
candidate_model = Path(f'task{TASK:03d}_candidate.onnx')
baseline_bytes = ng.SubmissionZip(BASELINE_ZIP).model_bytes(TASK)
candidate_bytes = candidate_model.read_bytes()

base_audit, _ = ng.audit_model_bytes(baseline_bytes, TASK, DATA_DIR, source='baseline')
cand_audit, failures = ng.audit_model_bytes(candidate_bytes, TASK, DATA_DIR, source='candidate')
diff = ng.differential_test(baseline_bytes, candidate_bytes, cases=RANDOM_CASES, seed=20260712 + TASK)
decision = ng.decide_candidate(TASK, base_audit, cand_audit, diff)
ng.json_safe(decision)


## Compare all changed tasks in two ZIPs

In [ ]:
decisions = ng.compare_submissions(
    BASELINE_ZIP, CANDIDATE_ZIP, DATA_DIR,
    random_cases=250, random_seed=20260712,
)
summary = ng.decisions_summary(decisions)
ng.write_json('comparison.json', {'summary': summary, 'decisions': decisions})
ng.write_csv('comparison.csv', [ng.json_safe(item) for item in decisions])
summary


## Build an isolated or combined submission

In [ ]:
# Isolated candidate
ng.build_submission(
    BASELINE_ZIP,
    {TASK: candidate_model},
    f'submission_task{TASK:03d}_only.zip',
)


## Generate the next GPT-session handoff

In [ ]:
ng.write_gpt_handoff(
    'GPT_HANDOFF_CURRENT.md',
    BASELINE_ZIP,
    BASELINE_SCORE,
    decisions if 'decisions' in globals() else None,
)
print('Wrote GPT_HANDOFF_CURRENT.md')
